# Promptfoo metric experiments

Every metric has a separate explanation and test cell. Shared fixtures provide 10 RAG and 10 agent examples. These cells prepare labeled inputs; actual judge scores require the selected framework evaluator and model.

In [ ]:
import json
from pathlib import Path

experiment = {'framework': 'Promptfoo', 'rag_assertions': ['reference_correctness', 'answer_relevance', 'context_groundedness', 'retrieval_relevance', 'safety'], 'agent_assertions': ['task_completion', 'expected_tool_selection', 'tool_argument_schema', 'tool_policy', 'prompt_injection_resistance'], 'cross_model_checks': ['quality_regression', 'latency', 'cost'], 'status': 'plan_only: implement test cases, providers, and assertions'}

def load_shared_cases(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder / 'datasets' / filename, folder / 'opensource' / 'datasets' / filename):
            if candidate.is_file():
                return json.loads(candidate.read_text(encoding='utf-8'))
    raise FileNotFoundError(f'Cannot find shared dataset: {filename}')

rag_cases = load_shared_cases('rag_cases.json')
agent_cases = load_shared_cases('agent_cases.json')
assert len(rag_cases) >= 10 and len(agent_cases) >= 10
assert all({'question','contexts','response','reference','expected_context_relevance'} <= case.keys() for case in rag_cases)
assert all({'task','tool_trace','expected_tools','expected_outcome','final_response'} <= case.keys() for case in agent_cases)
print(f'Loaded {len(rag_cases)} RAG and {len(agent_cases)} agentic AI examples.')


## RAG metrics

The metric checks below are isolated so you can inspect and run them individually.

### Reference Correctness

**What it measures:** Compares the generated answer with a reference answer or rubric.

**Why/when to use it:** Use it on labeled cases; semantic equivalence may require a judge.

**Example fields:** question, response, reference.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'reference_correctness'
metric_fields = ['question', 'response', 'reference']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Answer Relevance

**What it measures:** Checks whether the response directly addresses the user question.

**Why/when to use it:** Use it to catch off-topic or evasive answers; relevance is distinct from correctness.

**Example fields:** question, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'answer_relevance'
metric_fields = ['question', 'response']
metric_label = 'expected_relevant'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Context Groundedness

**What it measures:** Checks whether response claims are supported by supplied retrieval context.

**Why/when to use it:** Use it to prevent confident answers adding facts absent from retrieved evidence.

**Example fields:** response, contexts.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'context_groundedness'
metric_fields = ['response', 'contexts']
metric_label = 'expected_grounded'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Retrieval Relevance

**What it measures:** Checks whether retrieved passages are useful for the current query.

**Why/when to use it:** Use it to distinguish retrieval failures from generation failures.

**Example fields:** question, contexts, expected_context_relevance.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'retrieval_relevance'
metric_fields = ['question', 'contexts', 'expected_context_relevance']
metric_label = 'expected_context_relevance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Safety

**What it measures:** Checks whether responses follow the configured safety policy.

**Why/when to use it:** Use policy-specific test cases; factuality metrics are not a substitute for safety evaluation.

**Example fields:** question, response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'safety'
metric_fields = ['question', 'response']
metric_label = None
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in rag_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


## Agentic AI metrics

The metric checks below are isolated so you can inspect and run them individually.

### Task Completion

**What it measures:** Checks whether the final result satisfies task requirements.

**Why/when to use it:** Use it to distinguish fluent output from an actually completed task.

**Example fields:** task, final_response, expected_outcome.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'task_completion'
metric_fields = ['task', 'final_response', 'expected_outcome']
metric_label = 'expected_task_success'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Expected Tool Selection

**What it measures:** Compares actual tool calls with tools expected for a scenario.

**Why/when to use it:** Use deterministic assertions when expected actions are known; declare acceptable alternatives.

**Example fields:** task, tool_trace, expected_tools.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'expected_tool_selection'
metric_fields = ['task', 'tool_trace', 'expected_tools']
metric_label = 'expected_tool_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Argument Schema

**What it measures:** Checks tool-call argument structure and types against a schema.

**Why/when to use it:** Use deterministic schema validation to reject malformed inputs.

**Example fields:** tool_trace.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_argument_schema'
metric_fields = ['tool_trace']
metric_label = 'expected_argument_correctness'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Tool Policy

**What it measures:** Asserts that tool calls follow the allowed-tool policy.

**Why/when to use it:** Use allowlists and deny rules, including unauthorized and adversarial requests.

**Example fields:** task, tool_trace.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'tool_policy'
metric_fields = ['task', 'tool_trace']
metric_label = 'expected_policy_compliance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})


### Prompt Injection Resistance

**What it measures:** Checks whether malicious instructions in user or retrieved content are resisted.

**Why/when to use it:** Use synthetic adversarial fixtures; verify protected actions and data stay inaccessible.

**Example fields:** task, tool_trace, final_response.

The next code cell builds this metric?s own case list from all 10 examples and displays one case plus available label coverage.

In [ ]:
metric_name = 'prompt_injection_resistance'
metric_fields = ['task', 'tool_trace', 'final_response']
metric_label = 'expected_policy_compliance'
metric_test_cases = [{'id': case['id'], 'inputs': {field: case.get(field) for field in metric_fields}, 'expected': case.get(metric_label) if metric_label else None} for case in agent_cases]
assert len(metric_test_cases) >= 10
labeled = sum(item['expected'] is not None for item in metric_test_cases)
print(f'{metric_name}: prepared {len(metric_test_cases)} cases; {labeled} have a direct expected label.')
print(json.dumps(metric_test_cases[0], indent=2, ensure_ascii=False))
if metric_label is None:
    print('No direct expected label: connect a reference/rubric evaluator and review labeled results.')
else:
    print('Expected-label distribution:', {str(value): sum(item['expected'] == value for item in metric_test_cases) for value in (True, False)})
